# 01 - Data Cleaning

**Goal:** check that the raw data can be trusted before we analyse it.

Checks: column types, missing values, duplicates, invalid amounts, label values.

**Dataset:** ULB Credit Card Fraud (Kaggle) - card transactions made by European cardholders over two days in September 2013.
`V1`-`V28` are anonymised PCA components, so we **do not** interpret them as merchants, locations or behaviours.

In [ ]:
import sys
sys.path.append("..")  # so we can import from src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.plots import set_style, save, LEGIT_COLOR, FRAUD_COLOR, MUTED_COLOR
set_style()
pd.set_option("display.float_format", "{:,.4f}".format)

In [ ]:
from src.preprocessing import load_raw, data_quality_report, clean, CLEAN_PATH

raw = load_raw()
raw.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0000,-1.3598,-0.0728,2.5363,1.3782,-0.3383,0.4624,0.2396,0.0987,0.3638,...,-0.0183,0.2778,-0.1105,0.0669,0.1285,-0.1891,0.1336,-0.0211,149.6200,0
1,0.0000,1.1919,0.2662,0.1665,0.4482,0.0600,-0.0824,-0.0788,0.0851,-0.2554,...,-0.2258,-0.6387,0.1013,-0.3398,0.1672,0.1259,-0.0090,0.0147,2.6900,0
2,1.0000,-1.3584,-1.3402,1.7732,0.3798,-0.5032,1.8005,0.7915,0.2477,-1.5147,...,0.2480,0.7717,0.9094,-0.6893,-0.3276,-0.1391,-0.0554,-0.0598,378.6600,0
3,1.0000,-0.9663,-0.1852,1.7930,-0.8633,-0.0103,1.2472,0.2376,0.3774,-1.3870,...,-0.1083,0.0053,-0.1903,-1.1756,0.6474,-0.2219,0.0627,0.0615,123.5000,0
4,2.0000,-1.1582,0.8777,1.5487,0.4030,-0.4072,0.0959,0.5929,-0.2705,0.8177,...,-0.0094,0.7983,-0.1375,0.1413,-0.2060,0.5023,0.2194,0.2152,69.9900,0


## Column types

In [ ]:
raw.dtypes.value_counts()

float64    30
int64       1
Name: count, dtype: int64

In [ ]:
raw[["Time", "Amount", "Class"]].describe()

,Time,Amount,Class
count,"284,807.0000","284,807.0000","284,807.0000"
mean,"94,813.8596",88.3496,0.0017
std,"47,488.1460",250.1201,0.0415
min,0.0000,0.0000,0.0000
25%,"54,201.5000",5.6000,0.0000
50%,"84,692.0000",22.0000,0.0000
75%,"139,320.5000",77.1650,0.0000
max,"172,792.0000","25,691.1600",1.0000


## Data-quality report

In [ ]:
report = data_quality_report(raw)
pd.Series(report)

rows                284807
columns                 31
missing_values           0
duplicate_rows        1081
negative_amounts         0
zero_amounts          1825
class_values        [0, 1]
dtype: object

**Findings**
- No missing values.
- Some rows are exact duplicates (same time, amount and all 28 components). We drop them so they are not double-counted.
- No negative amounts. Zero-amount transactions are kept - they are usually card-verification checks, not errors.
- The label `Class` only contains `0` (legitimate) and `1` (fraud).

In [ ]:
df = clean(raw)
print(f"Rows before: {len(raw):,}  |  after: {len(df):,}  |  removed: {len(raw) - len(df):,}")

Rows before: 284,807  |  after: 283,726  |  removed: 1,081


## Class balance after cleaning

In [ ]:
counts = df["Class"].value_counts().rename({0: "Legitimate", 1: "Fraud"})
summary = pd.DataFrame({"transactions": counts, "share_pct": 100 * counts / counts.sum()})
summary

,transactions,share_pct
Class,,
Legitimate,283253,99.8333
Fraud,473,0.1667


Fraud is extremely rare (well under 1%). This is the central challenge of the project: a model that says "legitimate" every time would look ~99.8% accurate and catch **zero** fraud.

In [ ]:
df.to_csv(CLEAN_PATH, index=False)
print(f"Saved cleaned data to {CLEAN_PATH}")

Saved cleaned data to /home/tayyabshaikh/Documents/Projects/credit-card-fraud-analysis/data/creditcard_clean.csv
